In [ ]:
import copy
import pathlib

from laserfarm import DataProcessing, MacroPipeline

# Macro-Pipeline AHN Workflow - Normalization

## Set input/output paths

In [ ]:
root_path = pathlib.Path("/project/lidarac/Share/users/student[ID]")

# input path (retiled files)
input_path = root_path / "retiled"

# output path (normalized files)
output_path = root_path / "normalized"

In [ ]:
tiles = list(input_path.glob("tile_*_*/"))
print("Normalizing {} tiles".format(len(tiles)))

## Setup Cluster

Setup Dask cluster used for the macro-pipeline calculation.

In [ ]:
from dask.distributed import Client

client = Client("tcp://10.0.0.52:33961")
client

## Normalization

Generate the normalized height for each point.

In [ ]:
# setup input dictionary to configure the normalization pipeline
normalization_input = {
    "setup_local_fs": {
        "input_folder": input_path.as_posix(),
        "output_folder": output_path.as_posix()
    },
    "load": {"attributes": "all"},
    # Filter out artificially high points - give overflow error when writing
    "apply_filter": {
        "filter_type": "select_below",
        "attribute": "z",
        "threshold": 10000.
    },
    # # filter point cloud using polygons
    # "apply_filter": {
    #     "filter_type": "select_polygon",
    #     "polygon_string": "/project/lidarac/Data/Cliptest/shapefile/Clip_shape_exploded.shp",
    #     "read_from_file": True
    # },
    "normalize": 1,
    "clear_cache" : {},
}

In [ ]:
macro = MacroPipeline()

In [ ]:
# add pipeline list to macro-pipeline object and set the corresponding labels
for tile in tiles:
    dp = DataProcessing(tile.name, label=tile.name)
    normalization_input_ = copy.deepcopy(normalization_input)
    normalization_input_["export_point_cloud"] = {
        "filename": "{}.laz".format(tile.name),
        "overwrite": True
    }
    dp.config(normalization_input_)
    macro.add_task(dp)

In [ ]:
macro.setup_cluster(cluster=client.scheduler.address)

In [ ]:
# run!
macro.run()

In [ ]:
# save outcome results
macro.print_outcome(to_file="normalize.out")

In [ ]:
assert not macro.get_failed_pipelines(), "Some of the tasks have failed!"

## Terminate cluster

In [ ]:
# client.close()
# macro.shutdown()